# 🧪 EXERCISE: Group Gaze Metrics

> **Instructions:** This notebook has intentional gaps — paths, loops, calculations and plot labels that you must complete. Look for `# TODO` and `???` markers. Run each cell in order once you have filled in the blanks.

## Learning objectives

- Configure multi-participant, multi-session SESSIONS list
- Complete a summary function computing accuracy and mean saccade metrics
- Build a grouped bar chart of saccade metrics across all participants
- Run Kruskal-Wallis non-parametric statistical tests across groups

---

# VISION Task — Group Gaze Metrics (All Patients, All Sessions)

Aggregates and compares eye-movement metrics across all participants and sessions.  
Identifies group-level patterns, outliers, and between-participant variability.

| Analysis | Content |
|---|---|
| **Group summary** | Mean/SD per metric across all participants |
| **Between-participant** | Variability in fixation, saccade, blink, gaze error |
| **Longitudinal group** | Session-to-session improvement across patients |
| **Statistical tests** | Kruskal-Wallis, Mann-Whitney, Spearman |

**SCOPE: all participants × all sessions**

---
## 0 Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from pathlib import Path
import msgpack, struct

sns.set_theme(style='whitegrid', palette='tab10', font_scale=1.15)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

# ── TODO 1: Define all participants and their recording sessions ──────────────
# SESSIONS is a list of tuples: (participant_id, session_label, path_to_folder)
# Add one entry per participant × session.
SESSIONS = [
    # ('JF',  'S001', Path('???')),
    # ('JF',  'S002', Path('???')),
    # ('MT',  'S001', Path('???')),
    # Add more participants here...
]

GAZE_TOPIC               = 'gaze.3d.01.'
VELOCITY_THRESHOLD_DEG_S = 30
MIN_SACCADE_DUR_S        = 0.010
MAX_SACCADE_DUR_S        = 0.100
BLINK_GROUP_GAP_S        = 0.075
POSITION_STEP_DEG        = 4.5


---
## 1 Shared Helper Functions

In [ ]:
def load_pldata(path):
    """Load a Pupil Labs .pldata file → list of (topic, dict) tuples."""
    records = []
    with open(path, 'rb') as f:
        unpacker = msgpack.Unpacker(f, raw=False, strict_map_key=False)
        for item in unpacker:
            if isinstance(item, (list, tuple)) and len(item) == 2:
                topic, payload = item
                rec = payload if isinstance(payload, dict) else (
                    msgpack.unpackb(payload, raw=False, strict_map_key=False)
                    if isinstance(payload, bytes) else None
                )
                if rec is not None:
                    records.append((str(topic), rec))
    return records


def parse_onset_label(label):
    """
    Parse VISION_task onset annotation.
    Format: trial_001_onset_ax0_pos3R_circle
    Returns dict or None.
    """
    m = re.match(r'trial_(\d+)_onset_ax(\d+)_pos(\d+)([LR])_(\w+)', label)
    if m:
        return {
            'trial_num'   : int(m.group(1)),
            'axis_deg'    : int(m.group(2)),
            'position_num': int(m.group(3)),
            'side'        : m.group(4),
            'shape'       : m.group(5),
        }
    return None


def parse_offset_label(label):
    """Parse offset annotation → dict with trial_num and outcome."""
    m = re.match(r'trial_(\d+)_offset_(\w+)', label)
    if m:
        return {'trial_num': int(m.group(1)), 'outcome': m.group(2)}
    return None


def detect_saccades(gaze_df,
                    vel_threshold  = VELOCITY_THRESHOLD_DEG_S,
                    min_dur_ms     = SACCADE_MIN_DURATION_MS,
                    max_dur_ms     = SACCADE_MAX_DURATION_MS,
                    min_confidence = MIN_GAZE_CONFIDENCE):
    g = gaze_df[gaze_df['confidence'] >= min_confidence].copy()
    if len(g) < 10:
        return pd.DataFrame()

    ts  = g['timestamp'].values
    xs  = g['x_deg'].values.astype(float)
    ys  = g['y_deg'].values.astype(float)

    # Gaussian smooth (sigma ≈ 5 ms)
    sr    = 1.0 / np.median(np.diff(ts)) if len(ts) > 1 else 200.0
    sigma = max(1, int(0.005 * sr))
    xs_sm = gaussian_filter1d(xs, sigma)
    ys_sm = gaussian_filter1d(ys, sigma)

    # Instantaneous velocity
    dt       = np.diff(ts)
    dx       = np.diff(xs_sm)
    dy       = np.diff(ys_sm)
    vel      = np.hypot(dx, dy) / np.where(dt > 0, dt, 1e-6)
    vel      = np.append(vel, vel[-1])

    above = vel >= vel_threshold
    events = []
    in_sacc = False
    start_idx = 0
    for i, a in enumerate(above):
        if a and not in_sacc:
            in_sacc   = True
            start_idx = i
        elif not a and in_sacc:
            in_sacc = False
            dur_ms  = (ts[i - 1] - ts[start_idx]) * 1000
            if min_dur_ms <= dur_ms <= max_dur_ms:
                amp = np.hypot(xs_sm[i-1] - xs_sm[start_idx],
                               ys_sm[i-1] - ys_sm[start_idx])
                events.append({
                    'timestamp'     : ts[start_idx],
                    'end_timestamp' : ts[i - 1],
                    'duration_ms'   : dur_ms,
                    'amplitude_deg' : amp,
                    'peak_vel_deg_s': vel[start_idx:i].max(),
                    'start_x'       : xs_sm[start_idx],
                    'start_y'       : ys_sm[start_idx],
                    'end_x'         : xs_sm[i - 1],
                    'end_y'         : ys_sm[i - 1],
                })
    return pd.DataFrame(events)


def build_blink_table(blink_df, group_gap_s=BLINK_GROUP_GAP_S):
    onsets  = blink_df[blink_df['type'] == 'onset'].sort_values('timestamp')
    offsets = blink_df[blink_df['type'] == 'offset'].sort_values('timestamp')
    if len(onsets) == 0:
        return pd.DataFrame()
    ts_arr = onsets['timestamp'].values
    grouped_onsets = []
    group_start = ts_arr[0]
    for i in range(1, len(ts_arr)):
        if ts_arr[i] - ts_arr[i - 1] > group_gap_s:
            grouped_onsets.append(group_start)
            group_start = ts_arr[i]
    grouped_onsets.append(group_start)

    rows = []
    off_ts = offsets['timestamp'].values
    for go in grouped_onsets:
        candidates = off_ts[off_ts > go]
        if len(candidates):
            t_off = candidates[0]
            rows.append({
                'timestamp'     : go,
                'end_timestamp' : t_off,
                'duration_ms'   : (t_off - go) * 1000,
            })
    return pd.DataFrame(rows)


def events_in_window(df, t_start, t_end, ts_col='timestamp'):
    return df[(df[ts_col] >= t_start) & (df[ts_col] <= t_end)]


def trial_fixation_metrics(fix_in):
    if len(fix_in) == 0:
        return {
            'fix_count': 0,
            'fix_dur_mean_ms': np.nan,
            'fix_dur_total_ms': 0,
            'fix_dispersion_mean': np.nan
        }

    # FILTER: minimum duration + minimum confidence
    fix_valid = fix_in[
        (fix_in['duration_ms'] >= MIN_FIX_DURATION_MS) &
        (fix_in['confidence']  >= MIN_GAZE_CONFIDENCE)
    ]

    if len(fix_valid) == 0:
        return {
            'fix_count': 0,
            'fix_dur_mean_ms': np.nan,
            'fix_dur_total_ms': 0,
            'fix_dispersion_mean': np.nan
        }

    return {
        'fix_count': len(fix_valid),
        'fix_dur_mean_ms': fix_valid['duration_ms'].mean(),
        'fix_dur_total_ms': fix_valid['duration_ms'].sum(),
        'fix_dispersion_mean': (
            fix_valid['dispersion'].mean()
            if 'dispersion' in fix_valid else np.nan
        ),
    }


def trial_saccade_metrics(sacc_in, t_start, target_x=None, target_y=None):
    if len(sacc_in) == 0:
        return {'sacc_count': 0, 'sacc_amp_mean_deg': np.nan,
                'sacc_peak_vel_mean': np.nan, 'sacc_dur_mean_ms': np.nan,
                'sacc_latency_ms': np.nan}
    metrics = {
        'sacc_count'       : len(sacc_in),
        'sacc_amp_mean_deg': sacc_in['amplitude_deg'].mean(),
        'sacc_peak_vel_mean': sacc_in['peak_vel_deg_s'].mean(),
        'sacc_dur_mean_ms' : sacc_in['duration_ms'].mean(),
        'sacc_latency_ms'  : (sacc_in['timestamp'].min() - t_start) * 1000,
    }
    if target_x is not None and len(sacc_in):
        first = sacc_in.sort_values('timestamp').iloc[0]
        dist_start = np.hypot(first['start_x'] - target_x, first['start_y'] - target_y)
        dist_end   = np.hypot(first['end_x']   - target_x, first['end_y']   - target_y)
        metrics['first_sacc_toward_target'] = int(dist_end < dist_start)
    return metrics


def trial_blink_metrics(blink_in):
    if len(blink_in) == 0:
        return {'blink_count': 0, 'blink_dur_mean_ms': np.nan}
    return {
        'blink_count': len(blink_in),
        'blink_dur_mean_ms': blink_in['duration_ms'].mean()
    }


def gaze_at_response(gaze_in, t_end, window_s=0.15):
    """
    Return median gaze position in the 150 ms window just before key press.
    Only uses high-confidence, on-screen gaze samples.
    """
    if t_end is None or np.isnan(t_end):
        return None
    window = gaze_in[
        (gaze_in['timestamp'] >= t_end - window_s) &
        (gaze_in['timestamp'] <= t_end) &
        (gaze_in['confidence'] >= MIN_GAZE_CONFIDENCE) &
        (gaze_in['norm_x'].between(0.0, 1.0)) &
        (gaze_in['norm_y'].between(0.0, 1.0))
    ]
    if len(window) == 0:
        return None
    return {
        'x_deg'     : window['x_deg'].median(),
        'y_deg'     : window['y_deg'].median(),
        'confidence': window['confidence'].mean(),
    }


def get_target_xy(axis_deg, position_num, side_char):
    side  = 1 if side_char == 'R' else -1
    ecc   = POSITIONS_DEG[position_num - 1]
    a_rad = np.radians(axis_deg)
    return side * ecc * np.cos(a_rad), side * ecc * np.sin(a_rad)

---
## 2 Load All Participants & Sessions

In [ ]:
def load_session(recording_dir, session_label):
    """Load one session folder → (gaze_df, fix_df, blink_df, ann_df, trials, trial_metrics)"""
    recording_dir = Path(recording_dir)
    result = {'label': session_label, 'dir': recording_dir}

    # Load .pldata files
    try:
        gaze_raw   = load_pldata(recording_dir / 'gaze.pldata')
        fix_raw    = load_pldata(recording_dir / 'fixations.pldata')
        blink_raw  = load_pldata(recording_dir / 'blinks.pldata')
        ann_raw    = load_pldata(recording_dir / 'annotation.pldata')
    except FileNotFoundError as e:
        print(f'  [{session_label}] Missing file: {e}')
        return None

    def is_valid_norm(rec):
        norm = rec.get('norm_pos', [0, 0])
        return -0.05 <= norm[0] <= 1.05 and -0.05 <= norm[1] <= 1.05

    # Gaze
    gaze_rows = []
    for topic, rec in gaze_raw:
        if not topic.startswith('gaze.'): continue
        if not is_valid_norm(rec): continue
        norm = rec.get('norm_pos', [0.5, 0.5])
        gaze_rows.append({
            'timestamp' : rec.get('timestamp', np.nan),
            'confidence': rec.get('confidence', 0.0),
            'norm_x'    : norm[0],
            'norm_y'    : norm[1],
            'x_deg'     : (norm[0] - 0.5) * SCREEN_W_DEG,
            'y_deg'     : (norm[1] - 0.5) * SCREEN_H_DEG,
        })
    gaze_df = pd.DataFrame(gaze_rows).sort_values('timestamp').reset_index(drop=True)

    # Fixations
    fix_rows = []
    for topic, rec in fix_raw:
        if not topic.startswith('fixations'): continue
        norm = rec.get('norm_pos', [0.5, 0.5])
        raw_dur = rec.get('duration', 0.0)
        dur_ms =  raw_dur * 1000 if 10 > raw_dur > 0.1 else raw_dur
        fix_rows.append({
            'timestamp'   : rec.get('timestamp', np.nan),
            'confidence'  : rec.get('confidence', 0.0),
            'norm_x'      : norm[0],
            'norm_y'      : norm[1],
            'x_deg'       : (norm[0] - 0.5) * SCREEN_W_DEG,
            'y_deg'       : (norm[1] - 0.5) * SCREEN_H_DEG,
            'duration_ms' : dur_ms,
            'dispersion'  : rec.get('dispersion', np.nan),
        })
    fix_df = pd.DataFrame(fix_rows)
    if not fix_df.empty:
        fix_df = fix_df.sort_values('timestamp').reset_index(drop=True)

    # Blinks
    blink_rows = []
    for topic, rec in blink_raw:
        blink_rows.append({
            'timestamp': rec.get('timestamp', np.nan),
            'type'     : rec.get('type', ''),
            'duration' : rec.get('duration', np.nan),
        })
    blink_df = pd.DataFrame(blink_rows).sort_values('timestamp').reset_index(drop=True)
    blinks = build_blink_table(blink_df)

    # Annotations
    ann_rows = []
    for topic, rec in ann_raw:
        ann_rows.append({
            'timestamp': rec.get('timestamp', np.nan),
            'label'    : rec.get('label', ''),
        })
    ann_df = pd.DataFrame(ann_rows).sort_values('timestamp').reset_index(drop=True)

    # Parse trials
    onsets  = ann_df[ann_df['label'].str.contains('_onset_', na=False)].copy()
    offsets = ann_df[ann_df['label'].str.contains('_offset_', na=False)].copy()
    trial_list = []
    for _, row in onsets.iterrows():
        parsed = parse_onset_label(row['label'])
        if parsed:
            parsed['t_start'] = row['timestamp']
            trial_list.append(parsed)
    trials = pd.DataFrame(trial_list)
    if len(trials) == 0:
        print(f'  [{session_label}] No trials found')
        return None

    # Add end times
    offset_lookup = {}
    for _, row in offsets.iterrows():
        parsed = parse_offset_label(row['label'])
        if parsed:
            tn = parsed['trial_num']
            if tn not in offset_lookup:
                offset_lookup[tn] = {'t_end': row['timestamp'], 'outcome': parsed['outcome']}
    trials['t_end']   = trials['trial_num'].map(lambda x: offset_lookup.get(x, {}).get('t_end', np.nan))
    trials['outcome'] = trials['trial_num'].map(lambda x: offset_lookup.get(x, {}).get('outcome', 'unknown'))

    # Per-trial metrics
    sacc_df = detect_saccades(gaze_df)
    metrics_list = []
    for _, trial in trials.iterrows():
        t0, t1 = trial['t_start'], trial.get('t_end', np.nan)
        if np.isnan(t1): continue
        fix_in   = events_in_window(fix_df, t0, t1)
        sacc_in  = events_in_window(sacc_df, t0, t1) if len(sacc_df) else pd.DataFrame()
        blink_in = events_in_window(blinks, t0, t1) if len(blinks) else pd.DataFrame()
        gaze_resp= gaze_at_response(gaze_df, t1)
        tx, ty   = get_target_xy(trial['axis_deg'], trial['position_num'], trial['side'])

        row_m = {
            'session'       : session_label,
            'trial_num'     : trial['trial_num'],
            'axis_deg'      : trial['axis_deg'],
            'position_num'  : trial['position_num'],
            'side'          : trial['side'],
            'shape'         : trial.get('shape', ''),
            'outcome'       : trial['outcome'],
            't_start'       : t0,
            't_end'         : t1,
            'duration_s'    : t1 - t0,
            'target_x_deg'  : tx,
            'target_y_deg'  : ty,
            'eccentricity_deg': POSITION_STEP_DEG * trial['position_num'],
            'axis_label'    : {0: 'Horizontal', 45: 'Diagonal BL→TR', 135: 'Diagonal TL→BR'}.get(trial['axis_deg'], str(trial['axis_deg'])),
            'hemifield'     : 'Right' if trial['side'] == 'R' else 'Left',
        }
        row_m.update(trial_fixation_metrics(fix_in))
        row_m.update(trial_saccade_metrics(sacc_in, t0, tx, ty))
        row_m.update(trial_blink_metrics(blink_in))
        if gaze_resp is not None:
            row_m['resp_gaze_x'] = gaze_resp['x_deg']
            row_m['resp_gaze_y'] = gaze_resp['y_deg']
            row_m['resp_gaze_conf'] = gaze_resp['confidence']
            row_m['gaze_error_deg'] = np.hypot(gaze_resp['x_deg'] - tx, gaze_resp['y_deg'] - ty)
        metrics_list.append(row_m)

    trial_metrics = pd.DataFrame(metrics_list)
    print(f'  [{session_label}] {len(trials)} trials, {len(trial_metrics)} with metrics')
    return {
        'label': session_label,
        'gaze_df': gaze_df, 'fix_df': fix_df, 'blinks': blinks,
        'ann_df': ann_df, 'trials': trials, 'trial_metrics': trial_metrics,
        'sacc_df': sacc_df,
    }

# Load all sessions
print(f'\nLoading {len(SESSIONS)} sessions for {PARTICIPANT_IDS}...')
sessions = []
for pid, lbl, d in SESSIONS:
    s = load_session(d, lbl)
    if s:
        s['trial_metrics']['participant'] = pid
        sessions.append(s)

print(f'\nLoaded {len(sessions)} sessions successfully')
all_tm = pd.concat([s['trial_metrics'] for s in sessions], ignore_index=True)
print(f'Total trials across all sessions: {len(all_tm)}')


---
## 3 Group Summary Table

In [ ]:
def participant_session_summary(tm_subset):
    """Return a one-row summary dict for a subset of trial_metrics."""
    n  = len(tm_subset)
    nc = (tm_subset['outcome'] == 'correct').sum()

    # ── TODO 2: Complete this function ───────────────────────────────────────
    # Compute:
    #   accuracy    = nc / n
    #   sacc_lat_m  = mean saccade latency (ms)
    #   sacc_amp_m  = mean saccade amplitude (deg)
    #   fix_count_m = mean fixation count per trial
    #
    # Return a dict with keys:
    #   'N', 'Accuracy', 'Sacc latency (ms)', 'Sacc amplitude (°)', 'Fixations/trial'

    # Your code here:
    return {
        'N'                 : n,
        'Accuracy'          : ???,
        'Sacc latency (ms)' : ???,
        'Sacc amplitude (°)': ???,
        'Fixations/trial'   : ???,
    }

# Build summary table
rows = []
for pid, lbl, d in SESSIONS:
    s = next((x for x in sessions if x['participant'] == pid and x['label'] == lbl), None)
    if s is None:
        continue
    row = participant_session_summary(s['trial_metrics'])
    row.update({'Participant': pid, 'Session': lbl})
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index(['Participant', 'Session'])
print('Group Summary Table')
print(summary_df.round(3).to_string())


---
## 4 Group-Level Fixation Metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = [('fix_count','Fixation count/trial'), ('fix_dur_mean_ms','Mean fix duration (ms)'), ('fix_dur_total_ms','Total fix time (ms)')]
colors = sns.color_palette('tab10', len(PARTICIPANT_IDS))
pid_list = sorted(all_tm['participant'].unique())

for ax, (col, label) in zip(axes, metrics):
    data_by_pid = [all_tm[all_tm['participant']==pid][col].dropna().values for pid in pid_list]
    bp = ax.boxplot(data_by_pid, labels=pid_list, patch_artist=True,
                    medianprops={'color':'black','lw':2})
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    # Overlay individual session means
    for pi, pid in enumerate(pid_list):
        sess_means = all_tm[all_tm['participant']==pid].groupby('session')[col].mean().values
        ax.scatter([pi+1]*len(sess_means), sess_means, color=colors[pi],
                   zorder=5, s=40, alpha=0.8, marker='D')
    ax.set_ylabel(label); ax.set_title(label)
    plt.setp(ax.get_xticklabels(), rotation=45)

fig.suptitle('Group — Fixation Metrics by Participant', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 5 Group-Level Saccade Metrics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
sacc_metrics = [
    ('sacc_count',         'Saccade count / trial'),
    ('sacc_amp_mean_deg',  'Mean saccade amplitude (°)'),
    ('sacc_peak_vel_mean', 'Mean peak velocity (°/s)'),
    ('sacc_latency_ms',    'First saccade latency (ms)'),
]

# ── TODO 3: Plot saccade metrics across participants ──────────────────────────
# For each metric (ax, (col, label)):
#   1. Compute the per-participant mean of `col` from all_tm
#      Hint: all_tm.groupby('participant')[col].mean()
#   2. Plot a horizontal bar chart (ax.barh) sorted by value
#   3. Add the participant ID as y-tick labels
#   4. Set ax.set_xlabel(label) and ax.set_title(label)
#
# Your code here:
for ax, (col, label) in zip(axes, sacc_metrics):
    pass   # ← replace with your implementation

plt.suptitle('Group-Level Saccade Metrics (per participant mean)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 6 Group-Level Gaze Error

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: gaze error per participant (box)
ax = axes[0]
data_by_pid = [all_tm[all_tm['participant']==pid]['gaze_error_deg'].dropna().values for pid in pid_list]
bp = ax.boxplot(data_by_pid, labels=pid_list, patch_artist=True, medianprops={'color':'black','lw':2})
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_ylabel('Gaze error (°)'); ax.set_title('Gaze error at key press by participant')
plt.setp(ax.get_xticklabels(), rotation=45)

# Panel B: mean gaze error per participant, coloured by session
ax = axes[1]
per_ps = all_tm.groupby(['participant','session'])['gaze_error_deg'].mean().reset_index()
for pi, pid in enumerate(pid_list):
    sub = per_ps[per_ps['participant']==pid].sort_values('session')
    ax.plot(sub['session'], sub['gaze_error_deg'], 'o-', color=colors[pi], lw=1.5, ms=7, label=pid)
ax.set_ylabel('Mean gaze error (°)'); ax.set_xlabel('Session')
ax.set_title('Gaze error trajectory per participant')
ax.legend(fontsize=9, title='Participant'); plt.setp(ax.get_xticklabels(), rotation=45)

fig.suptitle('Group — Gaze Accuracy', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 7 Metrics by Axis — Group Level

In [ ]:
axis_order = ['Horizontal', 'Diagonal BL→TR', 'Diagonal TL→BR']
fix_metrics = [('fix_count','Fixation count'),('sacc_latency_ms','Saccade latency (ms)'),
               ('gaze_error_deg','Gaze error (°)'),('duration_s','Response time (s)')]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()
for ax, (col, label) in zip(axes, fix_metrics):
    grp = all_tm.dropna(subset=[col]).groupby(['participant','axis_label'])[col].mean().reset_index()
    means = grp.groupby('axis_label')[col].mean().reindex(axis_order)
    sds   = grp.groupby('axis_label')[col].std().reindex(axis_order)
    x = np.arange(len(axis_order))
    ax.bar(x, means, yerr=sds, capsize=5, color=sns.color_palette('Set2',3),
           edgecolor='k', lw=0.8, alpha=0.85)
    # Overlay participant dots
    for pi, pid in enumerate(pid_list):
        vals = grp[grp['participant']==pid].set_index('axis_label')[col].reindex(axis_order)
        ax.scatter(x, vals.values, color=colors[pi], zorder=5, s=50, alpha=0.8, label=pid)
    ax.set_xticks(x); ax.set_xticklabels(axis_order, rotation=15)
    ax.set_ylabel(label); ax.set_title(f'{label} by axis (group mean ± SD)')
    ax.legend(fontsize=8, title='Participant')

fig.suptitle('Group — Metrics by Axis', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 8 Statistical Tests — Group Level

In [ ]:
print('=' * 60)
print('GROUP-LEVEL STATISTICAL TESTS')
print('=' * 60)

metrics_to_test = [
    ('fix_count',        'Fixation count / trial'),
    ('sacc_latency_ms',  'Saccade latency (ms)'),
    ('sacc_amp_mean_deg','Saccade amplitude (°)'),
    ('gaze_error_deg',   'Gaze error at key press (°)'),
]

# ── TODO 4: Run Kruskal-Wallis tests comparing participants ──────────────────
# For each metric:
#   - Group all_tm by 'participant' and collect a list of arrays (one per participant)
#   - Run stats.kruskal(*groups) — non-parametric comparison across >2 groups
#   - Print: metric name, H-statistic, p-value, significance marker
#
# Hint: groups = [grp[col].dropna().values for _, grp in all_tm.groupby('participant')]
#       H, p = stats.kruskal(*groups)

for col, label in metrics_to_test:
    if col not in all_tm.columns:
        continue
    # Your code here:
    pass


---
## 9 Export

In [ ]:
out_path = 'VISION_group_gaze_metrics_all.csv'
all_tm.to_csv(out_path, index=False)
print(f'Saved: {out_path}  ({len(all_tm)} rows, '
      f'{all_tm["participant"].nunique()} participants, '
      f'{all_tm["session"].nunique()} sessions)')

summary_path = 'VISION_group_gaze_summary_per_participant.csv'
per_participant.round(3).to_csv(summary_path, index=False)
print(f'Saved: {summary_path}')